# Run the 3G-Vietnam pipeline on Google Colab — **R runtime**

A pure-R notebook (no Python / rpy2). It runs, in order:

1. **`Cleaning OCI and CB data - district level.R`** → `Clean data/dist_3G.Rda`
2. **`Cleaning MCL data.R`** → `mcl_district_panel.csv`, `Clean data/mcl_main.Rda`

### Before you start
1. **Use the R runtime:** Runtime → Change runtime type → **Runtime type: R**, High-RAM (no GPU).
2. **Mount Google Drive:** click the **folder icon** in the left sidebar → **Mount Drive** → authorize. (Cell 1 also tries to mount it programmatically.)
3. Make sure the copy of the scripts on Drive is the **current** version — especially that `Cleaning OCI and CB data - district level.R` starts with the standalone `library(...)` + data-load header (so it runs without `Setting up.R`).
4. Run the cells top to bottom. Step 4 (OCI) takes ~30–60 min.

## 1. Mount Drive & point at the project

In [ ]:
# mount Drive if it isn't already, then set the working directory
if (!dir.exists("/content/drive/MyDrive")) {
  message("Mounting Google Drive ...")
  try(system('python3 -c "from google.colab import drive; drive.mount(\'/content/drive\')"'))
}

PROJECT_DIR <- "/content/drive/MyDrive/3G/"       # <<< your project folder
if (!dir.exists(PROJECT_DIR))
  stop("Not found: ", PROJECT_DIR,
       "\n-> Mount Drive (folder icon, left sidebar -> Mount Drive), then re-run this cell.")
setwd(PROJECT_DIR)
cat("Working directory:", getwd(), "\n\n")

# Linux is case-sensitive; the scripts mix 'Raw Data'/'Raw data' and 'Clean data'.
alias <- function(real, link) if (dir.exists(real) && !file.exists(link)) file.symlink(normalizePath(real), link)
alias("Raw Data","Raw data"); alias("Raw data","Raw Data")
alias("Clean data","Clean Data"); alias("Clean Data","Clean data")
if (!dir.exists("Clean data")) dir.create("Clean data")

need <- c("Cleaning OCI and CB data - district level.R", "Cleaning MCL data.R", "vn_district_match.R",
          "Raw Data/Vietnam_Cell_tower.csv",
          "Raw Data/VNShapefile/gadm41_VNM_shp/gadm41_VNM_2.shp",
          "Raw Data/LFS/lfs_dist_11.csv", "Clean data/district_controls_09.Rda")
cat("input check (MISSING = fix before running):\n")
for (f in need) cat(sprintf("  [%-7s] %s\n", ifelse(file.exists(f), "OK", "MISSING"), f))

## 2. Packages

The R runtime usually already has the geospatial stack. This installs anything missing, then checks each package loads.

In [ ]:
pkgs <- c("tidyverse","data.table","stringi","haven","readxl","lubridate",
          "sf","terra","exactextractr")
need <- setdiff(pkgs, rownames(installed.packages()))
if (length(need)) { cat("installing:", paste(need, collapse = ", "), "\n"); install.packages(need) }

ok <- suppressWarnings(suppressMessages(vapply(pkgs, requireNamespace, logical(1), quietly = TRUE)))
print(ok)
if (!all(ok))
  cat("\nMissing system libraries for:", paste(names(ok)[!ok], collapse = ", "),
      "\n-> run the next cell (apt) ONCE, then Runtime > Restart session, then re-run THIS cell.\n")

### 2b. Only if the cell above reported missing packages

Installs the system libraries `sf`/`terra` need. **After it finishes: Runtime → Restart session, then re-run cell 1 and cell 2.** (The restart is what keeps R's compression library consistent.)

In [ ]:
system("apt-get update -qq && apt-get install -y -qq libgdal-dev libgeos-dev libproj-dev libudunits2-dev")
cat("\nDone. Now: Runtime > Restart session, then re-run cell 1 and cell 2.\n")

## 3. Helper — run a project `.R` file with Colab-friendly paths

Sources a script but redirects its hardcoded Windows Overleaf figure folder to a local `./Figures`, and drops any Windows `setwd()`.

In [ ]:
run_r_script <- function(path) {
  stopifnot(file.exists(path))
  dir.create("Figures", showWarnings = FALSE)
  code <- readLines(path, encoding = "UTF-8", warn = FALSE)
  code <- gsub("C:/Users/Anri Sakakibara/Dropbox/Apps/Overleaf/3G in Vietnam/Figures/Descriptive Stats",
               "Figures", code, fixed = TRUE)
  code <- code[!grepl("^\\s*setwd\\s*\\(", code)]
  message("Running: ", path); t0 <- Sys.time()
  eval(parse(text = paste(code, collapse = "\n")), envir = globalenv())
  message("Finished ", path, " in ",
          round(as.numeric(difftime(Sys.time(), t0, units = "mins")), 1), " min")
}
cat("run_r_script() ready\n")

## 4. Build `dist_3G` (OCI + Collins Bartholomew) — ~30–60 min

Writes `Clean data/dist_3G.Rda` (+ `.dta`), `oci_dist_1017.*`, `cb_dist_1017.*`.

In [ ]:
run_r_script("Cleaning OCI and CB data - district level.R")

## 5. Build the MCL job-ad district panel

Needs `Clean data/dist_3G.Rda` (from step 4) and `Clean data/district_controls_09.Rda`.

In [ ]:
run_r_script("Cleaning MCL data.R")

## 6. Check outputs

In [ ]:
for (f in c("Clean data/dist_3G.Rda", "Clean data/dist_3G.dta",
            "Clean data/oci_dist_1017.Rda", "mcl_district_panel.csv",
            "Clean data/mcl_main.Rda")) {
  sz <- if (file.exists(f)) sprintf(" (%.1f MB)", file.info(f)$size / 1e6) else ""
  cat(ifelse(file.exists(f), "OK  ", "--- "), f, sz, "\n")
}